### load apdb

In [75]:
from apdb_min_freq.load_fmin_data import load_fmin_data
df_apdb_min_freq = load_fmin_data('./apdb_min_freq/min_freq_data.csv')

Loaded 34763 entries from ./apdb_min_freq/min_freq_data.csv.
Filtered 387 entries with fc2_error > 0.1.
Filtered 13588 entries with same mpid, kept the lowest min_freq.


### load split_data

In [76]:
import pickle
import pandas as pd

# Load any split
with open('./random_split.pkl', 'rb') as f:
    data_random = pickle.load(f)

# Load any split
with open('./ood_split.pkl', 'rb') as f:
    data_ood = pickle.load(f)

# Load any split
with open('./space_group_split.pkl', 'rb') as f:
    data_space_group = pickle.load(f)

### functions

In [77]:
def make_dedup_df(df_X, df_y,split_name):
    df_X_Y = pd.concat([df_X, df_y], axis=1)
    duplicates = df_X['mp_ids'][df_X['mp_ids'].duplicated()]

    duplicate_list = duplicates.unique().tolist()
    dup = duplicate_list[0]
    print(f"Example duplicate mp_id: {dup}")
    print(df_X_Y[df_X_Y['mp_ids'] == dup][[f'y_{split_name}_log_klat',f'y_{split_name}_klat']])
   
    df_X_Y_dedup = (
        df_X_Y
          .sort_values(['mp_ids', f'y_{split_name}_log_klat'], ascending=[True, False]) 
          .drop_duplicates(subset='mp_ids', keep='first')     
          .reset_index(drop=True)
    )
    print(f"Original size: {len(df_X_Y)}, Deduplicated size: {len(df_X_Y_dedup)}")
    print("Deduplicated example:")
    print(df_X_Y_dedup[df_X_Y_dedup['mp_ids'] == dup][[f'y_{split_name}_log_klat',f'y_{split_name}_klat']])
    return df_X_Y_dedup

In [78]:
def apdb_concat(data,df_apdb,split_name):

    X_data = data[f'X_{split_name}']
    Y_klat = data[f'y_{split_name}_klat']
    Y_log_klat = data[f'y_{split_name}_log_klat']

    df_x_prop = pd.DataFrame({key: X_data[key] for key in X_data.keys()})
    df_x_indices = pd.DataFrame(data[f'{split_name}_original_indices'])
    df_x_indices.rename(columns={0: f'{split_name}_original_indices'}, inplace=True)
    df_x_all = pd.concat([df_x_prop, df_x_indices], axis=1)

    df_y_log_klat = pd.DataFrame(Y_log_klat, columns=[f'y_{split_name}_log_klat'])
    df_y_log_klat.rename(columns={0: f'y_{split_name}_log_klat'}, inplace=True)
    df_y_klat = pd.DataFrame(Y_klat, columns=[f'y_{split_name}_klat'])
    df_y_klat.rename(columns={0: f'y_{split_name}_klat'}, inplace=True)
    df_y_all = pd.concat([df_y_log_klat, df_y_klat], axis=1)

    df_x_y_concat = make_dedup_df(df_x_all, df_y_all, split_name)

    x_y_mpid_list = list(df_x_y_concat['mp_ids'])
    print(f'Number of MPIDs in random split: {len(x_y_mpid_list)}')
    df_apdb_isin_xy = df_apdb[df_apdb['mpid'].isin(x_y_mpid_list)]
    print(f'Number of MPIDs in random split with min freq data: {len(df_apdb_isin_xy)}')
    df_apdb_isin_xy.reset_index(drop=False, inplace=True)
    df_apdb_isin_xy.rename(columns={'index': f'{split_name}_apdb_min_freq_indices'}, inplace=True)
    common_list = list(df_apdb_isin_xy['mpid'])
    df_x_y_common = df_x_y_concat[df_x_y_concat['mp_ids'].isin(common_list)]
    print(f'Number of MPIDs in random split after merging with min freq data: {len(df_x_y_common)}')
    df_common = pd.merge(df_x_y_common, df_apdb_isin_xy, left_on='mp_ids', right_on='mpid', how='inner')
    print(f'Final number of MPIDs in random split after merging with min freq data: {len(df_common)}')
    return df_common


In [79]:
def create_dict(data,split_name,df):
    top_level_cols = [
        f'{split_name}_original_indices',
        f'y_{split_name}_log_klat',
        f'y_{split_name}_klat',
        f'{split_name}_apdb_min_freq_indices',
    ]

    for col in top_level_cols:
        data[col] = df[col].tolist()

    train_cols = [c for c in df.columns if c not in top_level_cols]
    data[f'X_{split_name}'] = df[train_cols].to_dict(orient='list')
    
    return data


### run

In [80]:
df_test_random_common = apdb_concat(data_random, df_apdb_min_freq,'test')
print('---')
df_train_random_common = apdb_concat(data_random, df_apdb_min_freq,'train')

Example duplicate mp_id: mp-4117
     y_test_log_klat  y_test_klat
77          2.035637     7.657131
749         2.167156     8.733407
Original size: 1403, Deduplicated size: 1354
Deduplicated example:
     y_test_log_klat  y_test_klat
700         2.167156     8.733407
Number of MPIDs in random split: 1354
Number of MPIDs in random split with min freq data: 1347
Number of MPIDs in random split after merging with min freq data: 1347
Final number of MPIDs in random split after merging with min freq data: 1347
---
Example duplicate mp_id: mp-3552
      y_train_log_klat  y_train_klat
194           1.923848      6.847255
2909          1.972063      7.185487
Original size: 5563, Deduplicated size: 5412
Deduplicated example:
      y_train_log_klat  y_train_klat
2642          1.972063      7.185487
Number of MPIDs in random split: 5412
Number of MPIDs in random split with min freq data: 5394
Number of MPIDs in random split after merging with min freq data: 5394
Final number of MPIDs in random 

/tmp/ipykernel_1344983/888979548.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_apdb_isin_xy.rename(columns={'index': f'{split_name}_apdb_min_freq_indices'}, inplace=True)
/tmp/ipykernel_1344983/888979548.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_apdb_isin_xy.rename(columns={'index': f'{split_name}_apdb_min_freq_indices'}, inplace=True)


In [81]:
df_test_ood_common = apdb_concat(data_ood, df_apdb_min_freq,'test')
print('---')    
df_train_ood_common = apdb_concat(data_ood, df_apdb_min_freq,'train')

Example duplicate mp_id: mp-7211
     y_test_log_klat  y_test_klat
66         -0.140784     0.868677
975        -0.090865     0.913141
Original size: 1914, Deduplicated size: 1852
Deduplicated example:
      y_test_log_klat  y_test_klat
1606        -0.090865     0.913141
Number of MPIDs in random split: 1852
Number of MPIDs in random split with min freq data: 1846
Number of MPIDs in random split after merging with min freq data: 1846
Final number of MPIDs in random split after merging with min freq data: 1846
---


/tmp/ipykernel_1344983/888979548.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_apdb_isin_xy.rename(columns={'index': f'{split_name}_apdb_min_freq_indices'}, inplace=True)


Example duplicate mp_id: mp-3552
      y_train_log_klat  y_train_klat
238           1.923848      6.847255
2711          1.972063      7.185487
Original size: 5042, Deduplicated size: 4909
Deduplicated example:
      y_train_log_klat  y_train_klat
2419          1.972063      7.185487
Number of MPIDs in random split: 4909
Number of MPIDs in random split with min freq data: 4890
Number of MPIDs in random split after merging with min freq data: 4890
Final number of MPIDs in random split after merging with min freq data: 4890


/tmp/ipykernel_1344983/888979548.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_apdb_isin_xy.rename(columns={'index': f'{split_name}_apdb_min_freq_indices'}, inplace=True)


In [82]:
df_test_space_group_common = apdb_concat(data_space_group, df_apdb_min_freq,'test')
print('---')
df_train_space_group_common = apdb_concat(data_space_group, df_apdb_min_freq,'train')

Example duplicate mp_id: mp-10096
     y_test_log_klat  y_test_klat
181         0.158112     1.171297
813         0.163127     1.177187
Original size: 1393, Deduplicated size: 1348
Deduplicated example:
   y_test_log_klat  y_test_klat
5         0.163127     1.177187
Number of MPIDs in random split: 1348
Number of MPIDs in random split with min freq data: 1343
Number of MPIDs in random split after merging with min freq data: 1343
Final number of MPIDs in random split after merging with min freq data: 1343
---


/tmp/ipykernel_1344983/888979548.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_apdb_isin_xy.rename(columns={'index': f'{split_name}_apdb_min_freq_indices'}, inplace=True)


Example duplicate mp_id: mp-3552
      y_train_log_klat  y_train_klat
186           1.923848      6.847255
2884          1.972063      7.185487
Original size: 5573, Deduplicated size: 5418
Deduplicated example:
      y_train_log_klat  y_train_klat
2631          1.972063      7.185487
Number of MPIDs in random split: 5418
Number of MPIDs in random split with min freq data: 5398
Number of MPIDs in random split after merging with min freq data: 5398
Final number of MPIDs in random split after merging with min freq data: 5398


/tmp/ipykernel_1344983/888979548.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_apdb_isin_xy.rename(columns={'index': f'{split_name}_apdb_min_freq_indices'}, inplace=True)


### create pkl

In [83]:
dict_random_w_apdp = {}
dict_random_w_apdp['split_type'] = data_random['split_type']
dict_random_w_apdp = create_dict(dict_random_w_apdp,'train',df_train_random_common)
dict_random_w_apdp = create_dict(dict_random_w_apdp,'test',df_test_random_common)
order = list(data_random.keys())
dict_random_w_apdp = {k: dict_random_w_apdp[k] for k in order if k in dict_random_w_apdp}
dict_random_w_apdp.keys()

dict_keys(['split_type', 'X_train', 'X_test', 'y_train_log_klat', 'y_test_log_klat', 'y_train_klat', 'y_test_klat', 'train_original_indices', 'test_original_indices'])

In [84]:
dict_ood_w_apdp = {}
dict_ood_w_apdp['split_type'] = data_ood['split_type']
dict_ood_w_apdp = create_dict(dict_ood_w_apdp,'train',df_train_ood_common)
dict_ood_w_apdp = create_dict(dict_ood_w_apdp,'test',df_test_ood_common)
order = list(data_ood.keys())
dict_ood_w_apdp = {k: dict_ood_w_apdp[k] for k in order if k in dict_ood_w_apdp}
dict_ood_w_apdp.keys()

dict_keys(['split_type', 'X_train', 'X_test', 'y_train_log_klat', 'y_test_log_klat', 'y_train_klat', 'y_test_klat', 'train_original_indices', 'test_original_indices'])

In [87]:
dict_space_group_w_apdb = {}
dict_space_group_w_apdb['split_type'] = data_space_group['split_type']
dict_space_group_w_apdb = create_dict(dict_space_group_w_apdb,'train',df_train_space_group_common)
dict_space_group_w_apdb = create_dict(dict_space_group_w_apdb,'test',df_test_space_group_common)
order = list(data_space_group.keys())
dict_space_group_w_apdb = {k: dict_space_group_w_apdb[k] for k in order if k in dict_space_group_w_apdb}
dict_space_group_w_apdb.keys()

dict_keys(['split_type', 'X_train', 'X_test', 'y_train_log_klat', 'y_test_log_klat', 'y_train_klat', 'y_test_klat', 'train_original_indices', 'test_original_indices'])

### save pkl

In [ ]:
import pickle
import gzip

# random_split
with gzip.open('random_split_dedup_w_min_freq.pkl.gz', 'wb') as f:
    pickle.dump(dict_random_w_apdp, f, protocol=pickle.HIGHEST_PROTOCOL)

# ood_split
with gzip.open('ood_split_dedup_w_min_freq.pkl.gz', 'wb') as f:
    pickle.dump(dict_ood_w_apdp, f, protocol=pickle.HIGHEST_PROTOCOL)

with gzip.open('space_group_split_dedup_w_min_freq.pkl.gz', 'wb') as f:
    pickle.dump(dict_space_group_w_apdb, f, protocol=pickle.HIGHEST_PROTOCOL)